# **Baseline Notebook**

---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd
from utstd.folders import *
from utstd.ipyrenders import *
at = AtFolder(course_code=36106, assignment='AT3')
at.run()
import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 87.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
group_name = "Neural Minds"
student_name = "Pal Patel"
student_id = "26173702"

In [3]:
# Do not modify
print_tile(size='h1', key='group_name', value=group_name)

In [4]:
# Do not modify
print_tile(size='h1', key='student_name', value=student_name)

In [5]:
# Do not modify
print_tile(size='h1', key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

In [6]:
!pip install -q scikit-learn

### 0.b Import Packages

In [7]:
import pandas as pd
import altair as alt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

---
## A. Assess Baseline Model

In [8]:
# DO NOT MODIFY THE CODE IN THIS CELL
try:
    X_train = pd.read_csv(at.folder_path / 'X_train.csv')
    X_val   = pd.read_csv(at.folder_path / 'X_val.csv')
    X_test  = pd.read_csv(at.folder_path / 'X_test.csv')
    print('X_train:', X_train.shape, '| X_val:', X_val.shape, '| X_test:', X_test.shape)
except Exception as e:
    print(e)

X_train: (177, 14) | X_val: (38, 14) | X_test: (38, 14)


### A.1 Generate Predictions with Baseline Model

In [9]:
# Baseline: K-Means k=2 (simplest possible clustering)
baseline_kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
baseline_kmeans.fit(X_train)
train_labels = baseline_kmeans.labels_
val_labels   = baseline_kmeans.predict(X_val)
print('Training cluster distribution:')
print(pd.Series(train_labels).value_counts().sort_index())
print('Validation cluster distribution:')
print(pd.Series(val_labels).value_counts().sort_index())

Training cluster distribution:
0    143
1     34
Name: count, dtype: int64
Validation cluster distribution:
0    33
1     5
Name: count, dtype: int64


### A.2 Selection of Performance Metrics

> The silhouette score was selected because it measures how well-separated and compact the clusters are. It is appropriate for clustering problems as it helps evaluate whether products are correctly grouped based on similarity in pricing and cost characteristics.

In [10]:
train_sil = silhouette_score(X_train, train_labels)
val_sil   = silhouette_score(X_val, val_labels)
train_db  = davies_bouldin_score(X_train, train_labels)
val_db    = davies_bouldin_score(X_val, val_labels)
print('=== BASELINE PERFORMANCE ===')
print(f'Silhouette Score  - Train: {train_sil:.4f} | Val: {val_sil:.4f}')
print(f'Davies-Bouldin    - Train: {train_db:.4f} | Val: {val_db:.4f}')
print(f'Inertia           - {baseline_kmeans.inertia_:.2f}')

=== BASELINE PERFORMANCE ===
Silhouette Score  - Train: 0.3614 | Val: 0.3369
Davies-Bouldin    - Train: 1.1601 | Val: 0.9673
Inertia           - 1745.15


In [11]:
performance_metrics_explanations = """
Three complementary metrics evaluate clustering quality:

1. SILHOUETTE SCORE (primary metric): Range -1 to 1. Above 0.5 = good clustering.
   Captures both cohesion (within-cluster tightness) and separation between clusters.
   Chosen as primary metric because it is intuitive and widely used for K-Means evaluation.

2. DAVIES-BOULDIN SCORE (secondary metric): Lower is better. 0 = perfect separation.
   Penalises clusters that are too close relative to their internal spread.

3. INERTIA (diagnostic): Total within-cluster sum of squared distances to centroids.
   Used to build the elbow curve for selecting optimal k in the final clustering notebook.
"""

In [12]:
# Do not modify
print_tile(size='h3', key='performance_metrics_explanations', value=performance_metrics_explanations)

### A.3 Baseline Model Performance

> The baseline K-Means model successfully grouped products into two clusters based on pricing and cost features. The results show reasonable cluster separation and provide useful insights into low-priced and premium product groups. However, the model is limited by using only two clusters, so further optimisation will be explored in the final model.

In [13]:
X_vis = X_train.copy()
X_vis['cluster'] = train_labels.astype(str)
alt.Chart(X_vis).mark_point(size=60, opacity=0.7).encode(
    x=alt.X('log_avg_price:Q', title='Log Avg Price (standardised)'),
    y=alt.Y('margin_pct:Q', title='Margin % (standardised)'),
    color=alt.Color('cluster:N', title='Cluster')
).properties(title='Baseline K-Means (k=2): Price vs Margin', width=450, height=350)

alt.Chart(...)

In [14]:
centroids = pd.DataFrame(baseline_kmeans.cluster_centers_, columns=X_train.columns)
centroids.index.name = 'cluster'
print('Cluster Centroids:')
centroids.round(3)

Cluster Centroids:


,log_avg_cost,cost_std,cost_change,log_avg_price,price_std,price_change,margin_pct,price_to_cost_ratio,num_offers,avg_discount_pct,max_discount_pct,cost_tier,is_discounted,price_stability_score
cluster,,,,,,,,,,,,,,
0,-0.207,-0.428,-0.431,-0.206,-0.373,-0.384,0.142,0.152,0.001,0.076,0.066,-0.222,0.007,-0.376
1,0.871,1.802,1.811,0.865,1.569,1.613,-0.598,-0.638,-0.006,-0.318,-0.277,0.934,-0.028,1.581


In [15]:
baseline_performance_explanations = """
Baseline K-Means with k=2 produces two broad clusters:
  Cluster 0: Lower-cost lower-price products (accessories, components)
  Cluster 1: Higher-cost higher-price products (complete bikes)

Silhouette around 0.3-0.5 for k=2 suggests structure exists but clustering is too coarse.
Cannot distinguish mid-range from premium products within either cluster.
Davies-Bouldin below 1.0 indicates reasonable separation for 2 broad clusters.

Limitation: k=2 is too simplistic for actionable business use.
The final model uses elbow method and silhouette analysis to find the optimal k.
This baseline sets a performance floor the final model must clearly exceed.
"""

In [16]:
# Do not modify
print_tile(size='h3', key='baseline_performance_explanations', value=baseline_performance_explanations)